# Phase 7: Context-Dependency Analysis

**Goal:** Analyze if article context affects comment toxicity

**This notebook includes:**
- 7.1: Article Topic Extraction
- 7.2: Duplicate Analysis
- 7.3: Context Interaction Analysis


## Setup: Import Libraries

In [2]:
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

print('✓ All libraries imported successfully!')

✓ All libraries imported successfully!


---
# Part 7.1: Article Topic Extraction
---

## 7.1.1: Load Data

In [3]:
# Load comment data
df_comments = pd.read_csv('bert_topic_assignment_with_outliers.csv')

print(f'✓ Loaded {len(df_comments):,} comments')
print(f'Unique article IDs: {df_comments["article_id"].nunique()}')

article_ids = df_comments['article_id'].unique()
print(f'\nArticles to categorize: {len(article_ids)}')

✓ Loaded 4,992 comments
Unique article IDs: 2688

Articles to categorize: 2688


In [ ]:
# Load article data from SOCC dataset

try:
    df_articles = pd.read_csv('gnm_articles.csv')  
    print(f'✓ Articles data loaded: {df_articles.shape}')
    print(f'Columns: {list(df_articles.columns)}')
    
    # Filter to only articles in our comment dataset
    df_articles = df_articles[df_articles['article_id'].isin(article_ids)]
    print(f'\n✓ Filtered to {len(df_articles)} relevant articles')
    
except FileNotFoundError:
    print('ERROR: gnm_articles.csv not found!')
    print('Please update the path above to point to your SOCC dataset')
    print('\nCreating dummy dataset for demonstration...')
    
    df_articles = pd.DataFrame({
        'article_id': article_ids,
        'title': ['Article ' + str(int(aid)) for aid in article_ids]
    })

✓ Articles data loaded: (10339, 8)
Columns: ['article_id', 'title', 'article_url', 'author', 'published_date', 'ncomments', 'ntop_level_comments', 'article_text']

✓ Filtered to 2688 relevant articles


## 7.1.2: Define Topic Categories

In [5]:
# Define categories and keywords
TOPIC_KEYWORDS = {
    'immigration': ['immigra', 'refugee', 'border', 'asylum', 'deportation', 'visa', 'migrant'],
    'politics': ['election', 'parliament', 'minister', 'vote', 'campaign', 'party', 'trudeau', 'harper'],
    'economy': ['economy', 'economic', 'budget', 'deficit', 'tax', 'gdp', 'inflation', 'financial'],
    'technology': ['tech', 'digital', 'internet', 'app', 'software', 'google', 'cyber', 'online'],
    'healthcare': ['health', 'medical', 'hospital', 'doctor', 'patient', 'medicare', 'disease'],
    'environment': ['climate', 'environment', 'carbon', 'pollution', 'renewable', 'emission'],
    'education': ['education', 'school', 'university', 'student', 'teacher', 'college', 'tuition'],
    'terrorism_security': ['terror', 'security', 'attack', 'isis', 'extremist', 'radical', 'violence'],
    'religion': ['religio', 'church', 'muslim', 'christian', 'islam', 'jewish', 'faith', 'mosque'],
    'social_issues': ['rights', 'equality', 'discrimination', 'racism', 'gender', 'lgbtq', 'diversity'],
    'sports': ['sport', 'hockey', 'soccer', 'football', 'baseball', 'nhl', 'nfl', 'olympic'],
    'crime_law': ['crime', 'police', 'law', 'court', 'judge', 'trial', 'prison', 'arrest']
}

print('Defined topic categories:')
for cat, keywords in TOPIC_KEYWORDS.items():
    print(f'  {cat}: {len(keywords)} keywords')

Defined topic categories:
  immigration: 7 keywords
  politics: 8 keywords
  economy: 8 keywords
  technology: 8 keywords
  healthcare: 7 keywords
  environment: 6 keywords
  education: 7 keywords
  terrorism_security: 7 keywords
  religion: 8 keywords
  social_issues: 7 keywords
  sports: 8 keywords
  crime_law: 8 keywords


## 7.1.3: Categorize Articles

In [6]:
def categorize_article(title, text=None):
    """Categorize article based on keyword matching"""
    content = str(title).lower()
    if text and pd.notna(text):
        content += ' ' + str(text).lower()
    
    scores = {}
    for category, keywords in TOPIC_KEYWORDS.items():
        score = 0
        for keyword in keywords:
            score += len(re.findall(r'\b' + keyword, content))
        scores[category] = score
    
    return max(scores, key=scores.get) if max(scores.values()) > 0 else 'other'

# Apply categorization
if 'article_text' in df_articles.columns:
    print('Using both title and article text for categorization...')
    df_articles['article_topic_category'] = df_articles.apply(
        lambda row: categorize_article(row['title'], row['article_text']), axis=1)
else:
    print('Using only title for categorization...')
    df_articles['article_topic_category'] = df_articles['title'].apply(categorize_article)

print('\n✓ Categorization complete!')
print(f'\nCategory distribution:')
display(df_articles['article_topic_category'].value_counts())

Using both title and article text for categorization...

✓ Categorization complete!

Category distribution:


article_topic_category
politics              1009
economy                338
crime_law              249
education              187
technology             165
healthcare             152
environment            135
terrorism_security     123
religion               119
immigration             95
social_issues           65
sports                  37
other                   14
Name: count, dtype: int64

## 7.1.4: Review Sample Categorizations

In [7]:
print('Sample categorizations (2 per category):')
print('='*80)

for category in df_articles['article_topic_category'].unique()[:5]:  # Show first 5
    print(f'\n{category.upper()}:')
    samples = df_articles[df_articles['article_topic_category'] == category].head(2)
    for idx, row in samples.iterrows():
        title = row['title'][:80] if 'title' in row else 'No title'
        print(f'  - {title}...')

Sample categorizations (2 per category):

POLITICS:
  - The Tories deserve another mandate - Stephen Harper doesn't...
  - Harper hysteria a sign of closed liberal minds...

RELIGION:
  - Fifty years in Canada, and now I feel like a second-class citizen...
  - A niqab ban makes no sense. Religious freedom is citizenship...

HEALTHCARE:
  - A nation of $100,000 firefighters...
  - Freed Canadians are radical grandstanders...

CRIME_LAW:
  - Our duty is to stand firm in the face of Russian aggression...
  - How U.S. gun ownership became a 'right,' and why it isn't...

ENVIRONMENT:
  - Sorry, pundits of Canada. The Leap will bring us together...
  - Whatever happened to global warming?...


## 7.1.5: Merge with Comment Data

In [8]:
# Create mapping
article_mapping = df_articles[['article_id', 'article_topic_category']].copy()

# Merge with comments
df = df_comments.merge(article_mapping, on='article_id', how='left')

print('✓ Merged comment data with article categories')
print(f'Shape: {df.shape}')
print(f'\nComments with category: {df["article_topic_category"].notna().sum():,}')
print(f'Comments missing category: {df["article_topic_category"].isna().sum():,}')

✓ Merged comment data with article categories
Shape: (4992, 9)

Comments with category: 4,992
Comments missing category: 0


## 7.1.6: Summary Statistics

In [9]:
category_stats = df.groupby('article_topic_category').agg({
    'comment_id': 'count',
    'article_id': 'nunique',
    'toxicity_level': 'mean'
}).round(3)

category_stats.columns = ['n_comments', 'n_articles', 'mean_toxicity']
category_stats = category_stats.sort_values('n_comments', ascending=False)

print('SUMMARY BY ARTICLE CATEGORY')
print('='*80)
display(category_stats)

print(f'\nMost commented: {category_stats.index[0]} ({category_stats.iloc[0]["n_comments"]:,.0f} comments)')
print(f'Highest toxicity: {category_stats["mean_toxicity"].idxmax()} (mean={category_stats["mean_toxicity"].max():.3f})')

SUMMARY BY ARTICLE CATEGORY


,n_comments,n_articles,mean_toxicity
article_topic_category,,,
politics,1986,1009,1.272
economy,616,338,1.213
crime_law,436,249,1.283
education,361,187,1.235
environment,297,135,1.219
healthcare,277,152,1.199
technology,263,165,1.238
religion,209,119,1.388
immigration,185,95,1.238



Most commented: politics (1,986 comments)
Highest toxicity: social_issues (mean=1.410)


## 7.1.7: Save Results

In [10]:
article_mapping.to_csv('Output/phase7_article_topic_mapping.csv', index=False)
print('✓ SAVED: Output/phase7_article_topic_mapping.csv')

df.to_csv('Output/phase7_comments_with_context.csv', index=False)
print('✓ SAVED: Output/phase7_comments_with_context.csv')

category_stats.to_csv('Output/phase7_article_category_summary.csv')
print('✓ SAVED: Output/phase7_article_category_summary.csv')

if 'title' in df_articles.columns:
    df_articles[['article_id', 'title', 'article_topic_category']].to_csv(
        'Output/phase7_articles_with_categories.csv', index=False)
    print('✓ SAVED: Output/phase7_articles_with_categories.csv')

✓ SAVED: Output/phase7_article_topic_mapping.csv
✓ SAVED: Output/phase7_comments_with_context.csv
✓ SAVED: Output/phase7_article_category_summary.csv
✓ SAVED: Output/phase7_articles_with_categories.csv


---
# Part 7.2: Duplicate Analysis
---

## 7.2.1: Find Exact Duplicates

In [11]:
# Find comments with same text across different articles
duplicates = df[df.duplicated('comment_text', keep=False)].copy()
duplicates = duplicates[duplicates.groupby('comment_text')['article_id'].transform('nunique') > 1]

print('DUPLICATE COMMENT ANALYSIS')
print('='*80)
print(f'✓ Found {len(duplicates["comment_text"].unique())} unique duplicate comments')
print(f'Total duplicate instances: {len(duplicates):,}')
print(f'\nExample: Same comment appearing in multiple articles')
if len(duplicates) > 0:
    example = duplicates.groupby('comment_text').head(1).iloc[0]
    print(f'  Text: "{example["comment_text"][:100]}..."')
    group = duplicates[duplicates['comment_text'] == example['comment_text']]
    print(f'  Appears in {len(group)} different articles')

DUPLICATE COMMENT ANALYSIS
✓ Found 8 unique duplicate comments
Total duplicate instances: 22

Example: Same comment appearing in multiple articles
  Text: "good point..."
  Appears in 2 different articles


## 7.2.2: Calculate Toxicity Variance

In [12]:
variance_results = []

for text, group in duplicates.groupby('comment_text'):
    if len(group) > 1:
        variance_results.append({
            'comment_text': text[:100],  # Truncate for display
            'n_occurrences': len(group),
            'n_articles': group['article_id'].nunique(),
            'toxicity_mean': group['toxicity_level'].mean(),
            'toxicity_std': group['toxicity_level'].std(),
            'toxicity_min': group['toxicity_level'].min(),
            'toxicity_max': group['toxicity_level'].max(),
            'toxicity_range': group['toxicity_level'].max() - group['toxicity_level'].min(),
        })

variance_df = pd.DataFrame(variance_results)
variance_df = variance_df.sort_values('toxicity_range', ascending=False)

print('\nToxicity Variance for Duplicate Comments:')
print('='*80)
print(f'Top 10 comments with highest toxicity variance:')
display(variance_df.head(10)[['n_occurrences', 'toxicity_mean', 'toxicity_std', 'toxicity_range']])

high_variance_count = (variance_df['toxicity_range'] > 1.0).sum()
high_variance_pct = (high_variance_count / len(variance_df) * 100) if len(variance_df) > 0 else 0

print(f'\nComments with toxicity range > 1.0: {high_variance_count} ({high_variance_pct:.1f}%)')


Toxicity Variance for Duplicate Comments:
Top 10 comments with highest toxicity variance:


,n_occurrences,toxicity_mean,toxicity_std,toxicity_range
0,2,1.0,0.0,0.0
1,2,1.0,0.0,0.0
2,2,1.0,0.0,0.0
3,2,1.0,0.0,0.0
4,2,1.0,0.0,0.0
5,2,1.0,0.0,0.0
6,2,1.0,0.0,0.0
7,8,1.0,0.0,0.0



Comments with toxicity range > 1.0: 0 (0.0%)


## 7.2.3: Calculate ICC (Intraclass Correlation)

In [20]:
try:
    from pingouin import intraclass_corr
    
    # Prepare data for ICC - need balanced design
    # Only use duplicates that appear in at least 2 articles
    duplicate_counts = duplicates.groupby('comment_text')['article_id'].nunique()
    valid_duplicates = duplicate_counts[duplicate_counts >= 2].index
    duplicates_for_icc = duplicates[duplicates['comment_text'].isin(valid_duplicates)].copy()
    
    # Create numeric IDs for ICC calculation
    duplicates_for_icc['comment_id_numeric'] = duplicates_for_icc.groupby('comment_text').ngroup()
    duplicates_for_icc['article_id_numeric'] = duplicates_for_icc.groupby('article_id').ngroup()
    
    print(f'Using {len(valid_duplicates)} duplicate comments for ICC calculation')
    print(f'Total instances: {len(duplicates_for_icc)}')
    
    # Calculate ICC
    icc_result = intraclass_corr(
        data=duplicates_for_icc,
        targets='comment_id_numeric',
        raters='article_id_numeric',
        ratings='toxicity_level',
        nan_policy='omit'
    )
    
    # Get ICC2 score (Two-way random effects, absolute agreement)
    icc_score = icc_result.loc[icc_result['Type'] == 'ICC2', 'ICC'].values[0]
    
    print('\nINTRACLASS CORRELATION COEFFICIENT (ICC)')
    print('='*80)
    print(f'ICC Score: {icc_score:.3f}')
    print(f'\nInterpretation:')
    if icc_score < 0.5:
        print('  Poor consistency - Context STRONGLY matters')
    elif icc_score < 0.7:
        print('  Moderate consistency - Context MATTERS')
    elif icc_score < 0.9:
        print('  Good consistency - Context has SOME effect')
    else:
        print('  Excellent consistency - Context does NOT matter')
    print('='*80)
    
except Exception as e:
    icc_score = None
    print('\n⚠️ Could not calculate ICC')
    print(f'Error: {e}')
    print('\nNote: This can happen if:')
    print('  - Not enough duplicate comments across different articles')
    print('  - Data is too unbalanced')
    print('  - Pingouin not installed: pip install pingouin')
    print('\nYou can still proceed - ICC is just one indicator.')
    print('The duplicate variance analysis (Part 7.2.2) is more important.')

Using 8 duplicate comments for ICC calculation
Total instances: 22

⚠️ Could not calculate ICC
Error: Data must have at least 5 non-missing values.

Note: This can happen if:
  - Not enough duplicate comments across different articles
  - Data is too unbalanced
  - Pingouin not installed: pip install pingouin

You can still proceed - ICC is just one indicator.
The duplicate variance analysis (Part 7.2.2) is more important.


## 7.2.4: DECISION - Does Context Matter?

In [21]:
print('\n' + '='*80)
print('DECISION: DOES CONTEXT MATTER?')
print('='*80)

# Criteria
print(f'\nEvidence:')
print(f'  1. High variance comments (>1.0): {high_variance_pct:.1f}%')
if icc_score:
    consistency_level = 'Low' if icc_score < 0.7 else 'High'
    print(f'  2. ICC Score: {icc_score:.3f} ({consistency_level} consistency)')

# Decision logic
context_matters = False
if high_variance_pct > 30:
    context_matters = True
    print(f'\n  → High variance > 30% suggests context MATTERS')
if icc_score and icc_score < 0.7:
    context_matters = True
    print(f'  → ICC < 0.7 suggests context MATTERS')

print('\n' + '='*80)
if context_matters:
    print('FINAL DECISION: Context DOES significantly matter')
else:
    print('FINAL DECISION: Context DOES NOT significantly matter')
print('='*80)

if context_matters:
    print('\n✓ Proceed with deep context analysis in Part 7.3')
else:
    print('\n✓ Context effects are minimal - simplified analysis is sufficient')


DECISION: DOES CONTEXT MATTER?

Evidence:
  1. High variance comments (>1.0): 0.0%

FINAL DECISION: Context DOES NOT significantly matter

✓ Context effects are minimal - simplified analysis is sufficient


## 7.2.5: Save Duplicate Analysis Results

In [15]:
variance_df.to_csv('Output/phase7_duplicate_variance.csv', index=False)
print('✓ SAVED: Output/phase7_duplicate_variance.csv')

with open('Output/phase7_context_decision.txt', 'w') as f:
    decision = 'YES' if context_matters else 'NO'
    f.write(f'Context matters: {decision}\n')
    f.write(f'High variance %: {high_variance_pct:.1f}%\n')
    if icc_score:
        f.write(f'ICC Score: {icc_score:.3f}\n')
        
print('✓ SAVED: Output/phase7_context_decision.txt')

✓ SAVED: Output/phase7_duplicate_variance.csv
✓ SAVED: Output/phase7_context_decision.txt


---
# Part 7.3: Context Interaction Analysis
---

## 7.3.1: Topic-Context Toxicity Matrix

In [16]:
# Create matrix: Comment Topic × Article Context → Mean Toxicity
df_clean = df.dropna(subset=['article_topic_category'])

matrix = df_clean.groupby([
    'bertopic_topic_label',
    'article_topic_category'
])['toxicity_level'].mean().unstack()

print('TOPIC-CONTEXT TOXICITY MATRIX')
print('='*80)
print('Rows = Comment Topics | Columns = Article Contexts')
print('Values = Mean Toxicity\n')
display(matrix.round(2))

print('\nKey observations:')
print(f'  - Matrix size: {matrix.shape[0]} topics × {matrix.shape[1]} contexts')
print(f'  - Overall mean toxicity: {df_clean["toxicity_level"].mean():.3f}')
print(f'  - Range: {matrix.min().min():.3f} to {matrix.max().max():.3f}')

TOPIC-CONTEXT TOXICITY MATRIX
Rows = Comment Topics | Columns = Article Contexts
Values = Mean Toxicity



article_topic_category,crime_law,economy,education,environment,healthcare,immigration,other,politics,religion,social_issues,sports,technology,terrorism_security
bertopic_topic_label,,,,,,,,,,,,,
"Animals, Food, Meat Ethics & Environment",1.38,1.14,1.33,1.00,NaN,1.00,1.00,1.28,1.00,4.00,1.00,1.33,1.67
"Bullying, Civility & Editorial Disputes",1.00,2.00,1.00,NaN,NaN,3.00,NaN,1.33,NaN,NaN,NaN,NaN,1.33
Canada Identity / Provinces / General National Talk,1.14,1.23,1.00,1.00,1.00,1.00,1.00,1.62,1.00,1.00,1.00,1.00,1.00
"Canadian Culture, Language & Immigration Identity",2.00,1.00,1.25,1.00,2.00,1.22,NaN,1.14,1.00,2.00,NaN,NaN,2.00
Canadian Party Politics & Electoral Reform,NaN,1.00,1.00,NaN,NaN,NaN,NaN,1.09,NaN,NaN,NaN,NaN,1.00
"Cities, Transit & Urban Development",1.25,1.10,1.00,1.17,1.00,1.00,1.00,1.07,NaN,NaN,NaN,1.50,NaN
"Citizenship, Dual Citizenship & Niqab Ceremony",1.40,1.00,NaN,NaN,NaN,NaN,NaN,1.11,1.33,1.50,NaN,2.00,1.00
Climate Change & CO2 Debate,1.00,1.50,1.00,1.12,1.00,NaN,NaN,1.07,1.00,1.00,NaN,NaN,NaN
Columnist/Political Culture Commentary (Coyne etc.),NaN,NaN,1.00,1.00,NaN,NaN,NaN,1.00,NaN,NaN,NaN,1.17,NaN



Key observations:
  - Matrix size: 47 topics × 13 contexts
  - Overall mean toxicity: 1.262
  - Range: 1.000 to 4.000


Some comment topics are "context amplifiers" - their toxicity changes dramatically based on the article:

Top 3 most context-sensitive topics:

Animals, Food, Meat Ethics - Ranges from 1.0 to 4.0 (+3.0 change!)

Toronto Politics - Ranges from 1.0 to 4.0 (+3.0 change!)

Race & Racism - Ranges from 1.0 to 3.0 (+2.0 change)

Statistical model results:
Overall R² = 0.1258 (topics + context explain 12.6% of toxicity)

Interaction IS significant - Topics and context DO interact

8 topics show "high amplification" (>1.0 point change across contexts)

## 7.3.2: Statistical Interaction Model

In [17]:
print('\nRunning interaction model...')
print('Model: Toxicity ~ CommentTopic + ArticleContext + (CommentTopic × ArticleContext)')

model = ols('''toxicity_level ~ 
               C(bertopic_topic_label) + 
               C(article_topic_category) + 
               C(bertopic_topic_label):C(article_topic_category)''', 
            data=df_clean).fit()

print('\nMODEL RESULTS')
print('='*80)
print(f'R²: {model.rsquared:.4f}')
print(f'Adjusted R²: {model.rsquared_adj:.4f}')
print(f'F-statistic: {model.fvalue:.4f}')
print(f'P-value: {model.f_pvalue:.6f}')

# Check interaction significance
interaction_pvalues = model.pvalues.filter(regex=':')
if len(interaction_pvalues) > 0:
    min_interaction_p = interaction_pvalues.min()
    interaction_significant = min_interaction_p < 0.05
    significance_text = 'SIGNIFICANT' if interaction_significant else 'NOT SIGNIFICANT'
    print(f'\nInteraction effect: {significance_text}')
    print(f'Min interaction p-value: {min_interaction_p:.6f}')
else:
    interaction_significant = False
    print('\nNo interaction terms found')

print('='*80)


Running interaction model...
Model: Toxicity ~ CommentTopic + ArticleContext + (CommentTopic × ArticleContext)

MODEL RESULTS
R²: 0.1258
Adjusted R²: 0.0543
F-statistic: 1.7587
P-value: 0.000000

Interaction effect: SIGNIFICANT
Min interaction p-value: 0.000003


## 7.3.3: Context Amplification Scores

In [18]:
# Calculate how much context affects each topic's toxicity
amplification_scores = []

for topic in df_clean['bertopic_topic_label'].unique():
    topic_data = df_clean[df_clean['bertopic_topic_label'] == topic]
    toxicity_by_context = topic_data.groupby('article_topic_category')['toxicity_level'].mean()
    
    if len(toxicity_by_context) > 1:
        amplification_scores.append({
            'topic': topic,
            'min_toxicity': toxicity_by_context.min(),
            'max_toxicity': toxicity_by_context.max(),
            'amplification': toxicity_by_context.max() - toxicity_by_context.min(),
            'contexts_measured': len(toxicity_by_context)
        })

amplification_df = pd.DataFrame(amplification_scores).sort_values('amplification', ascending=False)

print('CONTEXT AMPLIFICATION SCORES')
print('='*80)
print('Amplification = (Max toxicity across contexts) - (Min toxicity across contexts)')
print('\nTop 10 context-dependent topics:')
display(amplification_df.head(10))

print(f'\nTopics with high amplification (>1.0): {(amplification_df["amplification"] > 1.0).sum()}')

CONTEXT AMPLIFICATION SCORES
Amplification = (Max toxicity across contexts) - (Min toxicity across contexts)

Top 10 context-dependent topics:


,topic,min_toxicity,max_toxicity,amplification,contexts_measured
28,"Animals, Food, Meat Ethics & Environment",1.0,4.000000,3.000000,12
31,Toronto Politics (Rob Ford / Mayor Drama),1.0,4.000000,3.000000,9
6,Race & Racism Discourse (White/Black),1.0,3.000000,2.000000,10
27,Energy Tech (Solar/Wind/Nuclear/Power),1.0,3.000000,2.000000,8
45,Fascism/Nazism/Confederate Symbols,1.0,3.000000,2.000000,10
44,"Bullying, Civility & Editorial Disputes",1.0,3.000000,2.000000,6
30,"Gender, Feminism & Relationships",1.0,2.333333,1.333333,9
41,Insults / Ad Hominem Flame Replies,1.0,2.181818,1.181818,4
15,"Police, Guns & Crime",1.0,2.000000,1.000000,10
3,"Healthcare System: Doctors, Hospitals & Patients",1.0,2.000000,1.000000,9



Topics with high amplification (>1.0): 8


## 7.3.4: Save Context Analysis Results

In [19]:
matrix.to_csv('Output/phase7_topic_context_matrix.csv')
print('✓ SAVED: Output/phase7_topic_context_matrix.csv')

amplification_df.to_csv('Output/phase7_context_amplification.csv', index=False)
print('✓ SAVED: Output/phase7_context_amplification.csv')

with open('Output/phase7_interaction_model_summary.txt', 'w') as f:
    f.write(str(model.summary()))
print('✓ SAVED: Output/phase7_interaction_model_summary.txt')

✓ SAVED: Output/phase7_topic_context_matrix.csv
✓ SAVED: Output/phase7_context_amplification.csv
✓ SAVED: Output/phase7_interaction_model_summary.txt
